# Data Preparation and Corpus Building

Convert raw Amazon reviews data into enriched documents. Uses DuckDB SQL to sample data directly from compressed files without loading entire dataset into memory.

## Output Files
- **data/processed/corpus.pkl** - 20,000 enriched documents for indexing
- **data/processed/books_sample.parquet** - Structured data for application

## Workflow
1. Sample 20K reviews directly using SQL (no full load)
2. Sample metadata using SQL
3. Join in memory (only 20K records)
4. Clean and preprocess text
5. Build enriched documents
6. Save files

## 1. Setup Paths

In [1]:
from pathlib import Path
import sys

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir
sys.path.insert(0, str(project_root))

data_raw = project_root / "data" / "raw"
data_processed = project_root / "data" / "processed"
data_raw.mkdir(parents=True, exist_ok=True)
data_processed.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Output location: {data_processed}")

Project root: /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki
Output location: /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki/data/processed


## 2. Load Libraries and Check Data

In [2]:
import pandas as pd
import duckdb
import pickle
import string

reviews_file = data_raw / "Books.jsonl.gz"
metadata_file = data_raw / "meta_Books.jsonl.gz"

print(f"Reviews file exists: {reviews_file.exists()}")
print(f"Metadata file exists: {metadata_file.exists()}")

if not reviews_file.exists() or not metadata_file.exists():
    print("\nDownload from: https://amazon-reviews-2023.github.io/")
    print(f"Place in: {data_raw}")

Reviews file exists: True
Metadata file exists: True


## 3. Sample Reviews Directly

**What it does:** Uses DuckDB SQL to stratified sample 20K reviews directly from compressed file.

**Output:** Sample size and rating distribution

In [3]:
print("Sampling reviews directly from file (memory efficient)...")
print("This does NOT load entire file into memory.\n")

conn = duckdb.connect(':memory:')

# Sample reviews by rating to ensure balance
sample_query = f"""
WITH ranked_reviews AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY rating ORDER BY RANDOM()) as rn,
    COUNT(*) OVER (PARTITION BY rating) as rating_count
  FROM read_json_auto('{reviews_file}')
)
SELECT * EXCEPT (rn, rating_count)
FROM ranked_reviews
WHERE rn <= CAST(20000.0 * rating_count / SUM(rating_count) OVER () AS INTEGER)
LIMIT 20000
"""

try:
    reviews_df = conn.execute(sample_query).df()
    print(f"Sampled: {len(reviews_df):,} reviews")
    print(f"Rating distribution:")
    print(reviews_df['rating'].value_counts().sort_index())
except Exception as e:
    print(f"Sampling query failed. Using simpler approach...\n")
    # Fallback: simple random sample
    simple_query = f"SELECT * FROM read_json_auto('{reviews_file}') ORDER BY RANDOM() LIMIT 20000"
    reviews_df = conn.execute(simple_query).df()
    print(f"Sampled: {len(reviews_df):,} reviews (random)")
    print(f"Rating distribution:")
    print(reviews_df['rating'].value_counts().sort_index())

Sampling reviews directly from file (memory efficient)...
This does NOT load entire file into memory.

Sampling query failed. Using simpler approach...



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sampled: 20,000 reviews (random)
Rating distribution:
rating
1.0      887
2.0      770
3.0     1320
4.0     3120
5.0    13903
Name: count, dtype: int64


## 4. Sample Metadata Directly

In [4]:
print("\nLoading metadata (selective columns only)...")

# Load only needed columns
metadata_query = f"""
SELECT parent_asin, title, average_rating 
FROM read_json_auto('{metadata_file}')
"""

metadata_df = conn.execute(metadata_query).df()
print(f"Loaded: {len(metadata_df):,} products")


Loading metadata (selective columns only)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded: 4,448,181 products


## 5. Join Reviews with Metadata

**What it does:** Merges 20K reviews with product info.

**Output:** Joined row count and missing values

In [5]:
print("Joining reviews with metadata...")

metadata_subset = metadata_df[['parent_asin', 'title', 'average_rating']].copy()
metadata_subset.rename(columns={'title': 'product_title', 'average_rating': 'product_avg_rating'}, inplace=True)

merged_df = reviews_df.merge(metadata_subset, on='parent_asin', how='left')

print(f"Merged: {len(merged_df):,} rows")
print(f"Missing product titles: {merged_df['product_title'].isnull().sum()}")

Joining reviews with metadata...
Merged: 20,000 rows
Missing product titles: 0


## 6. Text Preprocessing

In [6]:
def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

print("Preprocessing text...")
merged_df['text_preprocessed'] = merged_df['text'].apply(preprocess_text)

before = len(merged_df)
merged_df = merged_df[merged_df['text_preprocessed'].str.len() >= 20]
after = len(merged_df)

print(f"Documents removed (< 20 chars): {before - after}")
print(f"Final count: {after:,} documents")

Preprocessing text...
Documents removed (< 20 chars): 1680
Final count: 18,320 documents


## 7. Build Enriched Corpus

**What it does:** Creates enriched documents with 6 fields: Book title, Product rating, Review rating, Verified purchase, Review text, Helpful votes.

**Output:** Corpus size and sample document

In [7]:
print("Building enriched corpus...")

corpus = []
for idx, row in merged_df.iterrows():
    doc = (
        f"Book: {row['product_title']}\n"
        f"Product Rating: {row['product_avg_rating']:.1f}/5 | "
        f"Review: {row['rating']}/5 | "
        f"Verified: {'Yes' if row['verified_purchase'] else 'No'}\n"
        f"{row['text']}"
    )
    if row['helpful_vote'] > 0:
        doc += f"\nHelpful: {row['helpful_vote']}"
    corpus.append(doc)

print(f"Corpus created: {len(corpus):,} documents")
print(f"\nSample document:")
print("-" * 70)
print(corpus[0][:400] + "...")
print("-" * 70)

Building enriched corpus...
Corpus created: 18,320 documents

Sample document:
----------------------------------------------------------------------
Book: Olive Trees and Honey: A Treasury of Vegetarian Recipes from Jewish Communities Around the World
Product Rating: 4.7/5 | Review: 3.0/5 | Verified: Yes
I haven't yet tried any of these recipes - most reviewers have commented that the recipes are wonderful - but I must say I was instantly disappointed in the appearance of the book - I really expect cookbooks, especially when they are about foo...
----------------------------------------------------------------------


## 8. Validate Corpus

In [8]:
lengths = [len(d) for d in corpus]
print(f"Total documents: {len(corpus):,}")
print(f"Document length - Min: {min(lengths):,} | Max: {max(lengths):,} | Avg: {sum(lengths)//len(corpus):,} chars")

test_tokens = preprocess_text(corpus[0]).split()
print(f"Tokenization test (first document): {len(test_tokens)} tokens")

Total documents: 18,320
Document length - Min: 90 | Max: 20,355 | Avg: 574 chars
Tokenization test (first document): 85 tokens


## 9. Save Files

In [9]:
print("Saving files...\n")

# Save corpus
corpus_path = data_processed / "corpus.pkl"
with open(corpus_path, "wb") as f:
    pickle.dump(corpus, f)
print(f"Saved: corpus.pkl")
print(f"  Path: {corpus_path}")
print(f"  Size: {corpus_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"  Documents: {len(corpus):,}")
print(f"  Used by: 03_bm25, 04_semantic, 06_rag notebooks")

print()

# Save structured data
parquet_path = data_processed / "books_sample.parquet"
app_data = merged_df[['product_title', 'text', 'rating', 'product_avg_rating', 'verified_purchase', 'helpful_vote', 'parent_asin']].copy()
app_data.to_parquet(parquet_path, index=False)
print(f"Saved: books_sample.parquet")
print(f"  Path: {parquet_path}")
print(f"  Size: {parquet_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"  Rows: {len(app_data):,}")
print(f"  Used by: Streamlit application")

print(f"\nComplete. Next: Run 03_bm25_keyword_search.ipynb")

Saving files...

Saved: corpus.pkl
  Path: /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/corpus.pkl
  Size: 10.14 MB
  Documents: 18,320
  Used by: 03_bm25, 04_semantic, 06_rag notebooks

Saved: books_sample.parquet
  Path: /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/books_sample.parquet
  Size: 6.00 MB
  Rows: 18,320
  Used by: Streamlit application

Complete. Next: Run 03_bm25_keyword_search.ipynb
